In [ ]:
!nvidia-smi

!pip install -q ultralytics
!pip install -q ultralytics albumentations opencv-python-headless

Thu Nov 12 21:09:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             27W /   70W |     661MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
from pathlib import Path
import cv2
import albumentations as A

# 126 original training images + 5 versions each = about 756 training images
AUGMENTATIONS_PER_IMAGE = 5

train_images_dir = Path("/content/runway_dataset/train/images")
train_labels_dir = Path("/content/runway_dataset/train/labels")

# Realistic camera changes.
# No flipping: arrows and sign directions must remain correct.
transform = A.Compose(
    [
        A.ShiftScaleRotate(
            shift_limit=0.06,
            scale_limit=0.15,
            rotate_limit=10,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.8
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.20,
            contrast_limit=0.15,
            p=0.7
        ),
        A.HueSaturationValue(
            hue_shift_limit=5,
            sat_shift_limit=15,
            val_shift_limit=15,
            p=0.4
        ),
        A.Perspective(
            scale=(0.02, 0.05),
            keep_size=True,
            p=0.3
        ),
        A.GaussianBlur(
            blur_limit=(3, 5),
            p=0.2
        ),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.50
    )
)

# Only select original photos, never augment an augmented image.
image_files = [
    p for p in train_images_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    and "_aug" not in p.stem
]

created = 0

for image_path in image_files:
    label_path = train_labels_dir / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    image_bgr = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    bboxes = []
    class_labels = []

    with open(label_path, "r") as f:
        for line in f:
            values = line.strip().split()

            if len(values) == 5:
                class_labels.append(int(values[0]))
                bboxes.append([float(x) for x in values[1:]])

    if not bboxes:
        continue

    for aug_number in range(1, AUGMENTATIONS_PER_IMAGE + 1):
        augmented = transform(
            image=image_rgb,
            bboxes=bboxes,
            class_labels=class_labels
        )

        # Skip an uncommon bad transformation with no visible sign.
        if len(augmented["bboxes"]) == 0:
            continue

        new_stem = f"{image_path.stem}_aug{aug_number}"
        new_image_path = train_images_dir / f"{new_stem}.jpg"
        new_label_path = train_labels_dir / f"{new_stem}.txt"

        augmented_bgr = cv2.cvtColor(
            augmented["image"],
            cv2.COLOR_RGB2BGR
        )
        cv2.imwrite(str(new_image_path), augmented_bgr)

        with open(new_label_path, "w") as f:
            for class_id, bbox in zip(
                augmented["class_labels"],
                augmented["bboxes"]
            ):
                f.write(
                    f"{class_id} "
                    f"{bbox[0]:.6f} {bbox[1]:.6f} "
                    f"{bbox[2]:.6f} {bbox[3]:.6f}\n"
                )

        created += 1

print(f"✅ Created {created} new augmented training images.")
print(f"✅ Training-image total: {len(list(train_images_dir.glob('*.*')))}")

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


✅ Created 629 new augmented training images.
✅ Training-image total: 755


In [ ]:
import zipfile
import os

# Updated to match the actual file present in your Colab files
zip_file_name = 'Airport Sign Labels.v2-five-signs-180-images-yolov8.yolov8.zip'
extract_path = '/content/runway_dataset'

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset successfully extracted to local SSD:", extract_path)
else:
    print(f"❌ Could not find '{zip_file_name}'. Make sure you uploaded it to the Colab file sidebar and that it includes '.zip' at the end!")

✅ Dataset successfully extracted to local SSD: /content/runway_dataset


In [ ]:
import yaml

yaml_path = '/content/runway_dataset/data.yaml'

# Load Roboflow's yaml
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Update root path for Colab local storage
config['path'] = '/content/runway_dataset'

# Save it back
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml configured successfully!")
print(f"Detected Sign Classes: {config.get('names')}")

✅ data.yaml configured successfully!
Detected Sign Classes: ['Runway_36_Direction_Right', 'Runway_36_Red', 'Taxiway_B', 'Taxiway_B_Direction_Left', 'Terminal_A_Direction_Right']


In [ ]:
from ultralytics import YOLO
import os

# Load pre-trained YOLO11 nano model
model_yolo11 = YOLO('yolo11n.pt')

results_yolo11 = model_yolo11.train(
    data='/content/runway_dataset/data.yaml',
    epochs=100,
    imgsz=416,
    batch=8,
    name='runway_signs_yolo11_nano_augmented',
    plots=True,

    # Image variation during training
    degrees=10,         # slight camera tilt
    translate=0.08,     # sign slightly off-center
    scale=0.20,         # sign nearer/farther from camera
    perspective=0.0005, # camera angle change
    hsv_h=0.01,         # very small color adjustment
    hsv_s=0.20,         # saturation adjustment
    hsv_v=0.25,         # brightness adjustment

    # Never flip signs with arrows/directions
    fliplr=0.0,
    flipud=0.0,

    # Keep training images realistic for individual signs
    mosaic=0.0
)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.2, hsv_v=0.25, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=runway_signs_yolo11_nano_augmented, nbs=64, nms=None, op

In [ ]:
# Evaluate performance of YOLO11 nano
metrics_yolo11 = model_yolo11.val()

print("\n" + "="*40)
print(f"YOLO11 Nano mAP50: {metrics_yolo11.box.map50:.4f}")
print(f"YOLO11 Nano mAP50-95: {metrics_yolo11.box.map:.4f}")
print("="*40)

# Download YOLO11 weights directly
from google.colab import files
yolo11_weights_path = 'runs/detect/runway_signs_yolo11_nano/weights/best.pt'

if os.path.exists(yolo11_weights_path):
    files.download(yolo11_weights_path)
    print("📥 Downloading your new YOLO11 best.pt weights to your computer...")
else:
    print("❌ YOLO11 weights not found. Verify training completed successfully.")

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1097.9±460.6 MB/s, size: 26.6 KB)
val: Scanning /content/runway_dataset/valid/labels.cache... 36 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 10.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.8it/s 0.8s
                   all         36         36      0.987          1      0.995      0.903
Runway_36_Direction_Right          8          8      0.986          1      0.995      0.925
         Runway_36_Red          6          6       0.98          1      0.995      0.866
             Taxiway_B         11         11      0.996          1      0.995      0.909
Taxiway_B_Direction_Left          6          6      0.988          1      0.995      0.912
Terminal_A_Direction_Right          5 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading your new YOLO11 best.pt weights to your computer...
